<a href="https://colab.research.google.com/github/D2718281828nis/BioMedAI/blob/main/example-MRI_slicer_viewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
!pip install pydicom numpy matplotlib scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 15.5 MB/s eta 0:00:00


In [ ]:
# Install required packages
!pip install zenodo_get nibabel -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.0 MB/s eta 0:00:00


In [16]:
# Install required packages
!pip install ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.8 MB/s eta 0:00:00


In [19]:
import numpy as np
import matplotlib.pyplot as plt
from pydicom import dcmread
from pydicom.data import get_testdata_file
import io
from PIL import Image
import nibabel as nib
import os
import zipfile
import pandas as pd
from ipywidgets import interact, IntSlider, Layout, VBox, HTML
import ipywidgets as widgets

In [20]:

# Function to properly perform MPR reconstruction
def multiplanar_reconstruction(volume):
    """
    Perform Multiplanar Reconstruction of a 3D volume
    Returns axial, coronal, and sagittal views
    Handles 4D data properly where the last dimension represents different images
    """
    # If we have 4D data (256, 256, 1, 500), reshape to 3D by creating depth from multiple axial slices
    if len(volume.shape) == 4:
        # For the synthetic data, we have (256, 256, 1, 500)
        # We'll stack multiple axial slices to create a 3D volume with depth
        # Take up to 50 slices for better visualization
        num_slices = min(50, volume.shape[3])

        # Create a 3D volume: (256, 256, num_slices)
        volume_3d = np.zeros((256, 256, num_slices))
        for i in range(num_slices):
            volume_3d[:, :, i] = volume[:, :, 0, i]  # Extract each 2D axial slice

        volume = volume_3d
    elif len(volume.shape) == 3:
        # If already 3D, just ensure correct orientation
        pass
    else:
        # For 2D, convert to 3D
        volume = np.stack([volume] * 10, axis=-1)  # Create 10 "slices"

    # Ensure we have a proper 3D volume for MPR
    if len(volume.shape) == 2:
        volume = np.stack([volume] * 10, axis=-1)

    # Axial view (z-slice) - original order [x, y, z]
    axial = volume  # Shape: (256, 256, depth) -> axial slices are [x, y, slice]

    # Coronal view (y-slice) - swap axes so y becomes the slice dimension
    # Original: [x, y, z] -> Transpose to [x, z, y] -> coronal slices are [x, z, y_slice]
    coronal = np.transpose(volume, (0, 2, 1))  # [x, z, y]

    # Sagittal view (x-slice) - swap axes so x becomes the slice dimension
    # Original: [x, y, z] -> Transpose to [z, y, x] -> sagittal slices are [z, y, x_slice]
    sagittal = np.transpose(volume, (2, 1, 0))  # [z, y, x]

    return axial, coronal, sagittal

# Function to visualize MPR with interactive slider and labels
def plot_mpr_interactive_with_labels(axial, coronal, sagittal, slice_num=0, labels_df=None):
    """
    Plot the three MPR planes with interactive slice selection and labels
    """
    # Ensure slice_num is within bounds
    max_slice = min(axial.shape[2], coronal.shape[2], sagittal.shape[2]) - 1
    slice_num = min(slice_num, max_slice)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Determine label for current slice
    label_text = "Unknown"
    if labels_df is not None and slice_num < len(labels_df):
        label_value = int(labels_df.iloc[slice_num]['Label'])  # Convert to int
        if label_value == 0:
            label_text = "Healthy Control (HC)"
        elif label_value == 1:
            label_text = "Alzheimer's Disease (AD)"

    # Axial view (top-down) - slice through the z-axis
    axes[0].imshow(axial[:, :, slice_num], cmap='gray', aspect='auto')
    axes[0].set_title(f'Axial View (Slice {slice_num})\n{label_text}', fontsize=12)
    axes[0].axis('off')

    # Coronal view (front-to-back) - slice through the y-axis
    axes[1].imshow(coronal[:, :, slice_num], cmap='gray', aspect='auto')
    axes[1].set_title(f'Coronal View (Slice {slice_num})\n{label_text}', fontsize=12)
    axes[1].axis('off')

    # Sagittal view (side view) - slice through the x-axis
    axes[2].imshow(sagittal[:, :, slice_num], cmap='gray', aspect='auto')
    axes[2].set_title(f'Sagittal View (Slice {slice_num})\n{label_text}', fontsize=12)
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# Function to load and process NIfTI MRI data
def load_nifti_mri(file_path):
    """
    Load a NIfTI format MRI file and return the 3D volume
    """
    img = nib.load(file_path)
    data = img.get_fdata()
    return data

# Enhanced Interactive MRI Viewer with labels
class EnhancedInteractiveMRIViewer:
    def __init__(self, volume, labels_df=None):
        self.volume = volume
        self.labels_df = labels_df
        self.axial, self.coronal, self.sagittal = multiplanar_reconstruction(volume)
        self.max_slice = min(self.axial.shape[2], self.coronal.shape[2], self.sagittal.shape[2]) - 1

    def show_interactive_view(self):
        """Show interactive viewer with slider and labels"""
        # Create the slider widget
        slice_slider = IntSlider(
            value=self.max_slice // 2,
            min=0,
            max=self.max_slice,
            step=1,
            description='Slice Number:',
            style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )

        # Create info text widget
        info_text = HTML(value=f"<b>Dataset Info:</b> Total slices: {self.max_slice + 1}<br>"
                              f"<b>Label Legend:</b> 0 = Healthy Control (HC), 1 = Alzheimer's Disease (AD)<br>"
                              f"<b>Note:</b> Each slice represents a central axial slice of a synthetic brain MRI")

        # Create the interactive widget
        interactive_plot = interact(
            lambda slice_num: plot_mpr_interactive_with_labels(
                self.axial, self.coronal, self.sagittal, slice_num, self.labels_df
            ),
            slice_num=slice_slider
        )

        # Combine info text and interactive plot
        combined_widget = VBox([info_text, interactive_plot.widget.children[-1]])
        return combined_widget

def list_files_in_directory(directory):
    """List all files in a directory with their paths"""
    for root, dirs, files in os.walk(directory):
        for file in files:
            print(os.path.join(root, file))

def find_nifti_files(directory):
    """Find all .nii and .nii.gz files in a directory and subdirectories"""
    nifti_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.nii') or file.endswith('.nii.gz'):
                nifti_files.append(os.path.join(root, file))
    return nifti_files

# Main execution
if __name__ == "__main__":
    print("=== Enhanced Interactive Brain MRI Multiplanar Reconstruction (MPR) in Google Colab ===")
    print("This code demonstrates how to process and visualize brain MRI data")
    print("using Multiplanar Reconstruction techniques with an interactive viewer and labels.\n")

    print("=== Using Brain MRI Dataset from Zenodo ===")
    print("Dataset: https://zenodo.org/records/8276786")
    print("This dataset contains synthetic brain MRI scans of Alzheimer's patients and healthy subjects.\n")
    print("Labels: 0 = Healthy Controls (HC), 1 = Alzheimer's Disease (AD)\n")

    print("=== Attempting to Download and Process Brain MRI Data ===")

    try:
        # The dataset contains Generated256x256.nii.gz and labels_syn.csv
        doi_id = "10.5281/zenodo.8276786"
        print(f"Accessing dataset with DOI: {doi_id}")

        # First, get the metadata to see what files are available
        import requests
        import json

        # Get the Zenodo record
        url = f"https://zenodo.org/api/records/8276786"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            print("Dataset metadata retrieved successfully!")

            # Look for files in the dataset
            files = data.get('files', [])
            print(f"Found {len(files)} files in the dataset:")
            for file_info in files:
                print(f"- {file_info['key']} ({file_info['size']})")

                # Download the main NIfTI file
                if 'Generated256x256.nii.gz' in file_info['key']:
                    print(f"Found the main NIfTI file: {file_info['key']}")

                    # Download the NIfTI file
                    file_url = file_info['links']['self']
                    print(f"Downloading {file_info['key']} from {file_url}...")

                    file_response = requests.get(file_url)
                    if file_response.status_code == 200:
                        with open(file_info['key'], 'wb') as f:
                            f.write(file_response.content)
                        print(f"Successfully downloaded {file_info['key']}")

                        # Also download the labels CSV file
                        labels_df = None
                        for label_file in files:
                            if 'labels_syn.csv' in label_file['key']:
                                label_url = label_file['links']['self']
                                print(f"Downloading {label_file['key']} from {label_url}...")
                                label_response = requests.get(label_url)
                                if label_response.status_code == 200:
                                    with open(label_file['key'], 'wb') as f:
                                        f.write(label_response.content)
                                    print(f"Successfully downloaded {label_file['key']}")

                                    # Load and display label information
                                    labels_df = pd.read_csv(label_file['key'])
                                    print(f"Labels shape: {labels_df.shape}")
                                    print("First few labels:")
                                    print(labels_df.head())
                                    print("\nLabel distribution:")
                                    print(labels_df['Label'].value_counts())

                        # Now load the NIfTI file
                        print(f"Loading NIfTI file: {file_info['key']}")
                        mri_volume = load_nifti_mri(file_info['key'])
                        print(f"Original MRI volume shape: {mri_volume.shape}")

                        # The volume has shape (256, 256, 1, 500) - 500 images of (256, 256, 1)
                        # Each image represents a central axial slice of a synthetic brain MRI
                        print("Processing 4D data to create 3D volumes for MPR...")

                        # Create a 3D volume by taking multiple images from the 4th dimension
                        # This will give us depth for proper MPR visualization
                        num_slices_to_use = min(50, mri_volume.shape[3])  # Use up to 50 slices

                        # Reshape to create a 3D volume: (256, 256, num_slices)
                        volume_3d = np.zeros((256, 256, num_slices_to_use))
                        for i in range(num_slices_to_use):
                            volume_3d[:, :, i] = mri_volume[:, :, 0, i]  # Extract each 2D slice

                        print(f"Reshaped volume for MPR: {volume_3d.shape}")

                        # Initialize the enhanced interactive MRI viewer with the reshaped 3D volume and labels
                        viewer = EnhancedInteractiveMRIViewer(volume_3d, labels_df)

                        print("Starting interactive viewer. Use the slider to change slice number...")
                        print("The label (HC/AD) for each slice will be displayed with the images.")

                        # Start the interactive viewer
                        interactive_widget = viewer.show_interactive_view()
                        display(interactive_widget)

                        print("\n=== How to Use the Interactive Viewer ===")
                        print("1. Use the slider to select different slice numbers")
                        print("2. The three views (axial, coronal, sagittal) update simultaneously")
                        print("3. Each view shows the same slice number from different orientations")
                        print("4. The label (HC/AD) for the current slice is displayed in the titles")

                    else:
                        print(f"Failed to download the NIfTI file. Status code: {file_response.status_code}")

                # Skip other files for now
                else:
                    print(f"Skipping file: {file_info['key']}")
        else:
            print(f"Failed to retrieve dataset metadata. Status code: {response.status_code}")
            print("Using synthetic data for demonstration...")

            # Create synthetic volume
            volume = create_sample_dicom_data()
            print(f"Synthetic volume shape: {volume.shape}")

            # Initialize the enhanced interactive MRI viewer with synthetic data
            viewer = EnhancedInteractiveMRIViewer(volume)
            print("Starting interactive viewer with synthetic data...")
            interactive_widget = viewer.show_interactive_view()
            display(interactive_widget)

    except Exception as e:
        print(f"An error occurred during processing: {str(e)}")
        print("This may happen if the dataset has access restrictions or requires special handling.")
        print("Using synthetic data for demonstration...")

        # Create synthetic volume
        volume = create_sample_dicom_data()
        print(f"Synthetic volume shape: {volume.shape}")

        # Initialize the enhanced interactive MRI viewer with synthetic data
        viewer = EnhancedInteractiveMRIViewer(volume)
        print("Starting interactive viewer with synthetic data...")
        interactive_widget = viewer.show_interactive_view()
        display(interactive_widget)

    print("\n=== Instructions for Processing Your Own Brain MRI Data ===")
    print("1. Search for brain MRI datasets on Zenodo (https://zenodo.org)")
    print("2. Identify a dataset DOI (e.g., 10.5281/zenodo.xxxxxxx)")
    print("3. Use zenodo_get to download the dataset")
    print("4. Extract the archive if needed")
    print("5. Find and load the .nii or .nii.gz files")
    print("6. Process with the EnhancedInteractiveMRIViewer class as shown above")

    print("\n=== About the Dataset Used ===")
    print("The dataset from https://zenodo.org/records/8276786 contains synthetic brain MRI scans")
    print("generated using PACGAN (Progressive Auxiliary Classifier Generative Adversarial Network).")
    print("These scans represent both healthy controls (Label 0) and Alzheimer's disease patients (Label 1).")
    print("Multiplanar Reconstruction allows visualization in axial, coronal, and sagittal planes.")
    print("The dataset includes 500 synthetic images with corresponding labels in labels_syn.csv.")
    print("Each image captures the central axial slice of a synthetic brain MRI.")

=== Enhanced Interactive Brain MRI Multiplanar Reconstruction (MPR) in Google Colab ===
This code demonstrates how to process and visualize brain MRI data
using Multiplanar Reconstruction techniques with an interactive viewer and labels.

=== Using Brain MRI Dataset from Zenodo ===
Dataset: https://zenodo.org/records/8276786
This dataset contains synthetic brain MRI scans of Alzheimer's patients and healthy subjects.

Labels: 0 = Healthy Controls (HC), 1 = Alzheimer's Disease (AD)

=== Attempting to Download and Process Brain MRI Data ===
Accessing dataset with DOI: 10.5281/zenodo.8276786
Dataset metadata retrieved successfully!
Found 2 files in the dataset:
- Generated256x256.nii.gz (104546782)
Found the main NIfTI file: Generated256x256.nii.gz
Successfully downloaded Generated256x256.nii.gz
Successfully downloaded labels_syn.csv
Labels shape: (500, 2)
First few labels:
   ID  Label
0   0      0
1   1      0
2   2      0
3   3      0
4   4      0

Label distribution:
Label
0    250
1 

interactive(children=(IntSlider(value=24, description='Slice Number:', layout=Layout(width='50%'), max=49, sty…


=== How to Use the Interactive Viewer ===
1. Use the slider to select different slice numbers
2. The three views (axial, coronal, sagittal) update simultaneously
3. Each view shows the same slice number from different orientations
4. The label (HC/AD) for the current slice is displayed in the titles
- labels_syn.csv (2899)
Skipping file: labels_syn.csv

=== Instructions for Processing Your Own Brain MRI Data ===
1. Search for brain MRI datasets on Zenodo (https://zenodo.org)
2. Identify a dataset DOI (e.g., 10.5281/zenodo.xxxxxxx)
3. Use zenodo_get to download the dataset
4. Extract the archive if needed
5. Find and load the .nii or .nii.gz files
6. Process with the EnhancedInteractiveMRIViewer class as shown above

=== About the Dataset Used ===
The dataset from https://zenodo.org/records/8276786 contains synthetic brain MRI scans
generated using PACGAN (Progressive Auxiliary Classifier Generative Adversarial Network).
These scans represent both healthy controls (Label 0) and Alzheim